#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [1]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangChainProject


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)



🔹 [AI 답변]:
## LangChain vs. LangGraph – 한눈에 보는 차이점

| 구분 | **LangChain** | **LangGraph** |
|------|----------------|---------------|
| **핵심 목표** | LLM‑기반 **애플리케이션**을 빠르게 구성·배포하기 위한 **프레임워크** | LLM‑기반 **워크플로우**를 **그래프(노드·엣지)** 형태로 정의·실행하기 위한 **라이브러리** |
| **주요 개념** | *Chains*, *Agents*, *Memory*, *PromptTemplates*, *Callbacks* | *Nodes*, *Edges*, *State*, *Conditional Logic*, *Loops* |
| **상태 관리** | 에이전트·체인 내부에서 메모리 객체(`ConversationBufferMemory`, `VectorStoreRetrieverMemory` 등) 사용 → **단일 스텝** 혹은 **연속적인** 호출에 적합 | **전역 상태(state) 객체**를 명시적으로 정의하고, 각 노드가 입·출력으로 상태를 읽고 쓰게 함 → **복잡한 상태 흐름**을 자연스럽게 표현 |
| **워크플로우 표현력** | 순차적 체인 + 간단한 조건부(If‑else)·루프(while) 구현은 가능하지만, 복잡한 DAG(Directed Acyclic Graph)·반복 구조는 코드로 직접 제어해야 함 | **그래프 구조** 자체가 설계 원칙 → 노드·엣지·조건부 전환을 선언적으로 정의 → 복수의 분기·합류·재귀가 자연스럽게 구현 |
| **디버깅·시각화** | 콜백을 이용해 로그를 남기지만, 전체 흐름을 한눈에 보는 도구는 제한적 | `graph.draw()` 같은 메서드로 **시각적인 DAG**를 자동 생성 → 실행 흐름을 UI/노트북에서 바로 확인 가능 |
| **재사용성** | 체인·프롬프트·메모리 등을 개별 컴포넌트로 재사용 가능 | **노드**를 독립적인 재사용 단위로 만든 뒤,